# Real-Time Data Ingestion with Spark Structured Streaming and PostgreSQL
**By:** Molo Munyansanga

This notebook runs the streaming pipeline step by step, so each stage can be seen:

1. **Part 1, the data:** look at the CSV files written by the data generator.
2. **Part 2, the transformations:** clean and validate a few files, one step at a time.
3. **Part 3, the stream:** start Spark Structured Streaming, watch it write to PostgreSQL in real time, and check the results.

The logic lives in the `src/` folder and is shared with `spark_streaming_to_postgres.py`, which runs the same pipeline from the terminal.

**You'll need two things running:**
- this notebook, for Spark;
- a second terminal for the data generator, `python data_generator.py`, started when the notebook asks for it.

> **Don't run `spark_streaming_to_postgres.py` while this notebook's stream is running.** Both would use the same checkpoint (Spark's record of which files it has processed).

## Setup

In [1]:
# Let the notebook find the src/ folder (one level above this notebook)
import sys
sys.path.append("..")

import pandas as pd
from pyspark.sql import functions as F

from src import monitor
from src.config import BATCH_LOG, GENERATOR_LOG, INCOMING_DIR
from src.postgres_sink import read_query, table_counts
from src.schemas import RAW_EVENT_SCHEMA
from src.spark_session import get_spark
from src.streaming import reset_pipeline, start_stream
from src.transform import add_reject_reason, clean_events, normalize, parse_types, rejected_events, valid_events

spark = get_spark()
# Show Spark DataFrames as tables in the notebook
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 40)
print(f"Spark {spark.version}")

Spark 4.3.0


### Start from a clean slate
`reset_pipeline()` empties the three tables and deletes the incoming files, Spark's checkpoint and both logs, so the results below come from this run only.

In [2]:
reset_pipeline()
table_counts()

{'events': 0, 'rejected_events': 0, 'events_staging': 0}

---
## Part 1: The data

**Now open a second terminal** (with the virtual environment activated) and write three files:

```
python data_generator.py --files 3
```

The next cell waits until the files arrive.

In [3]:
files = monitor.wait_for_files(min_files=3)

Waiting for at least 3 file(s) in incoming/ ...
Found 3 file(s).


Spark reads the files with an explicit schema. Every column is read as **text** on purpose: if Spark read `price` as a number, a value like `"abc"` would silently become null and the original value would be lost. Reading text first lets the cleaning step decide what is valid.

In [4]:
RAW_EVENT_SCHEMA.simpleString()

'struct<event_id:string,event_time:string,user_id:string,session_id:string,event_type:string,product_id:string,product_name:string,category:string,price:string,quantity:string>'

In [5]:
raw = (
    spark.read.schema(RAW_EVENT_SCHEMA).option("header", True).csv(str(INCOMING_DIR))
    .withColumn("source_file", F.col("_metadata.file_name"))
)
print(f"{raw.count()} rows in {len(files)} files")
raw.limit(5)

150 rows in 3 files


event_id,event_time,user_id,session_id,event_type,product_id,product_name,category,price,quantity,source_file
3dfdfe29-825b-41b8-8d56-4048c448a44a,2026-11-06 16:44:44.154,6658,605edd1b-68f1-405f-b2ad-06d7980d7dca,view,503,Children's Picture Book,Books,8.99,1,events_20261106_164439_00002.csv
5a43a40a-f770-4304-b8cf-99d949f89e0e,2026-11-06 16:44:44.154,8439,a462cbe0-1c05-45f6-82ab-cf84f2978776,view,503,Children's Picture Book,Books,8.99,1,events_20261106_164439_00002.csv
14c8b4b5-5d69-49df-b771-2f789cbfd08e,2026-11-06 16:44:44.154,9977,4c670178-5932-4917-8e18-ce73dbbe96c8,view,501,Mystery Novel,Books,11.99,many,events_20261106_164439_00002.csv
06cae91b-3564-4806-a234-55e1d9525e40,2026-11-06 16:44:44.154,6117,c665c230-25bf-4d72-a08d-11a5a302d278,View,503,Children's Picture Book,Books,8.99,1,events_20261106_164439_00002.csv
5100aa03-0d9d-4d61-ae3f-658c03ef5ff3,2026-11-06 16:44:44.155,2095,8b67ad65-c12d-45c4-9f67-e4e097155aba,view,101,Wireless Headphones,Electronics,89.99,1,events_20261106_164439_00002.csv


The generator deliberately includes three kinds of imperfect rows. Its log records how many of each it wrote, which is the answer key for checking the pipeline later:

In [6]:
pd.read_csv(GENERATOR_LOG)

,run_id,file_name,written_at,total_rows,valid_rows,messy_rows,invalid_rows,duplicate_rows
0,20261106_164439,events_20261106_164439_00001.csv,2026-11-06 16:44:39,50,47,1,2,1
1,20261106_164439,events_20261106_164439_00002.csv,2026-11-06 16:44:44,50,48,3,2,0
2,20261106_164439,events_20261106_164439_00003.csv,2026-11-06 16:44:49,50,44,2,5,1
3,20261106_164439,events_20261106_164439_00004.csv,2026-11-06 16:44:54,50,49,2,0,1


---
## Part 2: The transformations

The cleaning functions in `src/transform.py` work the same on a normal DataFrame and a streaming one, so they can be tried here on the three files first.

### Step 1: Normalize the formatting
Remove extra spaces, lowercase `event_type` and `event_id`, and turn empty text into null. Each cleaned value goes into a new `clean_` column, so the original stays available. Below are the **messy** rows, before and after:

In [7]:
normalized = normalize(raw)
messy = normalized.filter(
    (F.col("event_type") != F.col("clean_event_type")) | (F.col("product_name") != F.col("clean_product_name"))
)
messy.select("event_type", "clean_event_type", "product_name", "clean_product_name").limit(8)

event_type,clean_event_type,product_name,clean_product_name
View,view,Children's Picture Book,Children's Picture Book
VIEW,view,Denim Jacket,Denim Jacket
View,view,Cookbook,Cookbook
view,view,Desk Lamp,Desk Lamp
view,view,Coffee Maker,Coffee Maker
add_to_cart,add_to_cart,Bluetooth Speaker,Bluetooth Speaker


### Step 2: Convert to the right types
`try_cast` and `try_to_timestamp` return null for values that can't be converted (e.g. a quantity of `"two"`), instead of stopping the stream with an error. Spark 4 runs in ANSI mode, where a plain `cast` would fail the whole batch.

In [8]:
parsed = parse_types(normalized)
parsed.select("event_time", "parsed_event_time", "price", "parsed_price", "quantity", "parsed_quantity").limit(5)

event_time,parsed_event_time,price,parsed_price,quantity,parsed_quantity
2026-11-06 16:44:44.154,2026-11-06 16:44:44.154,8.99,8.99,1,1
2026-11-06 16:44:44.154,2026-11-06 16:44:44.154,8.99,8.99,1,1
2026-11-06 16:44:44.154,2026-11-06 16:44:44.154,11.99,11.99,many,NULL
2026-11-06 16:44:44.154,2026-11-06 16:44:44.154,8.99,8.99,1,1
2026-11-06 16:44:44.155,2026-11-06 16:44:44.155,89.99,89.99,1,1


### Step 3: Validate
Every row is checked against the rules below, and the **first** rule it breaks is recorded in `reject_reason` (null if it passes all of them). The rules match the constraints of the `events` table in PostgreSQL, so the database never receives a row it would refuse.

| Rule | Reason if broken |
|---|---|
| `event_id` present and a valid UUID | `missing_event_id` / `invalid_event_id` |
| `event_time` present and a valid timestamp | `missing_event_time` / `invalid_event_time` |
| `user_id` present and a whole number | `missing_user_id` / `invalid_user_id` |
| `session_id` present | `missing_session_id` |
| `event_type` is view, add_to_cart or purchase | `unknown_event_type` |
| `product_id` a whole number; `product_name`, `category` present | `invalid_product_id` / `missing_...` |
| `price` a number above 0 | `invalid_price` / `non_positive_price` |
| `quantity` a whole number above 0 | `invalid_quantity` / `non_positive_quantity` |

In [9]:
checked = add_reject_reason(parsed)
checked.groupBy("reject_reason").count().orderBy(F.desc("count"))

reject_reason,count
NULL,141
invalid_quantity,3
non_positive_price,3
invalid_event_time,2
missing_user_id,1


The rejected rows, with their original values:

In [10]:
rejected_events(checked).select("reject_reason", "event_time", "user_id", "event_type", "price", "quantity")

reject_reason,event_time,user_id,event_type,price,quantity
invalid_quantity,2026-11-06 16:44:44.154,9977,view,11.99,many
non_positive_price,2026-11-06 16:44:44.155,6585,view,-12.50,1
non_positive_price,2026-11-06 16:44:39.141,4841,view,-59.00,1
invalid_quantity,2026-11-06 16:44:39.143,3274,purchase,199.99,many
invalid_event_time,not-a-date,6585,add_to_cart,14.99,1
non_positive_price,2026-11-06 16:44:49.166,3650,view,-11.99,1
missing_user_id,2026-11-06 16:44:49.166,NULL,view,15.99,1
invalid_quantity,2026-11-06 16:44:49.168,8412,view,89.99,many
invalid_event_time,not-a-date,8260,view,89.00,1


### Step 4: Shape the valid rows for the `events` table
Valid rows get their final column names and types, plus `total_amount` (price × quantity). This is `clean_events()` followed by `valid_events()`, the same functions the stream uses.

In [11]:
valid = valid_events(clean_events(raw))
valid.printSchema()
valid.limit(5)

root
 |-- event_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- session_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: decimal(10,2) (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_amount: decimal(12,2) (nullable = true)
 |-- source_file: string (nullable = false)



event_id,event_time,user_id,session_id,event_type,product_id,product_name,category,price,quantity,total_amount,source_file
3dfdfe29-825b-41b8-8d56-4048c448a44a,2026-11-06 16:44:44.154,6658,605edd1b-68f1-405f-b2ad-06d7980d7dca,view,503,Children's Picture Book,Books,8.99,1,8.99,events_20261106_164439_00002.csv
5a43a40a-f770-4304-b8cf-99d949f89e0e,2026-11-06 16:44:44.154,8439,a462cbe0-1c05-45f6-82ab-cf84f2978776,view,503,Children's Picture Book,Books,8.99,1,8.99,events_20261106_164439_00002.csv
06cae91b-3564-4806-a234-55e1d9525e40,2026-11-06 16:44:44.154,6117,c665c230-25bf-4d72-a08d-11a5a302d278,view,503,Children's Picture Book,Books,8.99,1,8.99,events_20261106_164439_00002.csv
5100aa03-0d9d-4d61-ae3f-658c03ef5ff3,2026-11-06 16:44:44.155,2095,8b67ad65-c12d-45c4-9f67-e4e097155aba,view,101,Wireless Headphones,Electronics,89.99,1,89.99,events_20261106_164439_00002.csv
115a3706-1dd1-43db-bb00-b20045ac68f9,2026-11-06 16:44:44.155,9508,687900aa-edd2-471b-a83a-c5bb6666ccc6,view,503,Children's Picture Book,Books,8.99,1,8.99,events_20261106_164439_00002.csv


### Duplicates
Some valid rows are exact copies of earlier events. The stream removes duplicates within a batch with `dropDuplicates(["event_id"])`, and PostgreSQL skips any event already stored by an earlier batch (`ON CONFLICT DO NOTHING`), because `event_id` is the table's primary key.

In [12]:
duplicates = valid.groupBy("event_id").count().filter("count > 1")
print(f"{valid.count()} valid rows, {valid.select('event_id').distinct().count()} unique events, "
      f"{duplicates.count()} event(s) appearing more than once")

141 valid rows, 139 unique events, 2 event(s) appearing more than once


---
## Part 3: The stream

### Start the stream
`start_stream()` builds the streaming query and starts it **in the background**, so the notebook stays usable:

- **Source:** the `data/incoming/` folder. Every 5 seconds (the *trigger*), Spark takes the files that arrived since the last check as one **micro-batch** (at most 20 files at a time).
- **Processing and sink:** `foreachBatch` passes each micro-batch to `write_batch()` in `src/postgres_sink.py`, which cleans it with the functions above and writes it to PostgreSQL. Valid rows go through the staging table into `events`, and invalid rows go to `rejected_events`.
- **Checkpoint:** Spark records which files it has processed in `data/checkpoints/`, so a restarted stream continues where it stopped instead of processing everything again.

The three files from Part 1 are already in the folder, so they become the first micro-batch.

In [13]:
query = start_stream(spark)
query.status

{'message': 'Initializing sources',
 'isDataAvailable': False,
 'isTriggerActive': False}

### Watch it in real time
**Now start the generator in your second terminal, without a file limit:**

```
python data_generator.py
```

The next cell prints the stream's progress every 5 seconds for 40 seconds. You can run it again to keep watching.

In [14]:
monitor.watch(query, seconds=40, every=5)

time       batch   rows   rows/s   events  rejected  status
16:45:17       -      0        0        0         0  Initializing sources
16:45:22       -      0        0        0         0  Processing new data
16:45:27       0    200       22      188         9  Waiting for next trigger
16:45:33       0    200       22      188         9  Waiting for next trigger
16:45:38       0    200       22      188         9  Waiting for next trigger
16:45:43       1      0        0      188         9  Waiting for next trigger
16:45:48       1      0        0      188         9  Waiting for next trigger
16:45:53       1      0        0      188         9  Waiting for next trigger


### Inside the latest micro-batch
`query.lastProgress` is Spark's report on the most recent batch: how many rows it read (`numInputRows`), how fast (`processedRowsPerSecond`), and how long each part took (`durationMs`, in milliseconds).

In [15]:
progress = query.lastProgress
{key: progress[key] for key in ["batchId", "numInputRows", "inputRowsPerSecond", "processedRowsPerSecond", "durationMs"]}

{'batchId': 1,
 'numInputRows': 0,
 'inputRowsPerSecond': 0.0,
 'processedRowsPerSecond': 0.0,
 'durationMs': {'latestOffset': 8, 'triggerExecution': 11}}

Every batch is also recorded in `data/metrics/batch_log.csv` by `write_batch()`:

In [16]:
pd.read_csv(BATCH_LOG)

,batch_id,finished_at,duration_seconds,rows_read,valid_rows,duplicates_in_batch,inserted_rows,already_stored,rejected_rows
0,0,2026-11-06 16:45:25,6.447,200,191,3,188,0,9


### Look inside PostgreSQL
Spark can also *read* from PostgreSQL, so the tables can be queried while the stream is still writing to them.

In [17]:
read_query(spark, """
    SELECT event_time, user_id, event_type, product_name, price, quantity, total_amount, batch_id
    FROM events ORDER BY inserted_at DESC, event_time DESC LIMIT 10
""")

event_time,user_id,event_type,product_name,price,quantity,total_amount,batch_id
2026-11-06 16:44:54.181,5955,add_to_cart,Yoga Mat,25.00,1,25.00,0
2026-11-06 16:44:54.181,9379,view,Leather Wallet,35.00,1,35.00,0
2026-11-06 16:44:54.181,7793,view,Running Shoes,120.00,1,120.00,0
2026-11-06 16:44:54.181,9379,purchase,Mystery Novel,11.99,3,35.97,0
2026-11-06 16:44:54.181,3352,view,Data Engineering Guide,39.99,1,39.99,0
2026-11-06 16:44:54.181,6585,view,USB-C Charger,24.50,1,24.50,0
2026-11-06 16:44:54.181,1034,view,Wireless Headphones,89.99,1,89.99,0
2026-11-06 16:44:54.181,7409,add_to_cart,Denim Jacket,75.50,1,75.50,0
2026-11-06 16:44:54.181,7335,add_to_cart,Running Shoes,120.00,3,360.00,0
2026-11-06 16:44:54.181,1167,view,Smartphone Case,19.99,1,19.99,0


In [18]:
read_query(spark, """
    SELECT event_type, COUNT(*) AS events, SUM(total_amount) AS total_value
    FROM events GROUP BY event_type ORDER BY events DESC
""")

event_type,events,total_value
view,132,6217.190000000000000000
add_to_cart,39,4105.590000000000000000
purchase,17,1885.740000000000000000


In [19]:
read_query(spark, """
    SELECT reject_reason, COUNT(*) AS rows, MIN(event_type) AS example_type,
           MIN(price) AS example_price, MIN(quantity) AS example_quantity
    FROM rejected_events GROUP BY reject_reason ORDER BY rows DESC
""")

reject_reason,rows,example_type,example_price,example_quantity
invalid_quantity,3,purchase,11.99,many
non_positive_price,3,view,-11.99,1
invalid_event_time,2,add_to_cart,14.99,1
missing_user_id,1,view,15.99,1


A first look at **latency**, the time from an event happening (`event_time`) to its arrival in PostgreSQL (`inserted_at`). Performance is measured properly in Part 4.

In [20]:
read_query(spark, """
    SELECT ROUND(AVG(EXTRACT(EPOCH FROM inserted_at - event_time))::numeric, 2)::float AS avg_latency_seconds,
           ROUND(MAX(EXTRACT(EPOCH FROM inserted_at - event_time))::numeric, 2)::float AS max_latency_seconds
    FROM events
""")

avg_latency_seconds,max_latency_seconds
39.15,46.7


### Check the results against the answer key
**Stop the generator now** (press Ctrl+C in its terminal). The next cell waits until the stream has processed every row the generator wrote, then compares the tables with the generator's log.

In [21]:
monitor.wait_until_caught_up()

Caught up: 200 of 200 rows processed.


In [22]:
monitor.check_results()

[PASS] Valid events stored once each: expected 188, found 188
[PASS] Invalid rows rejected: expected 9, found 9
[PASS] Staging table emptied after each batch: expected 0, found 0

3 duplicate rows were skipped, and 8 messy rows were cleaned and stored.


True

### Stop the stream
A streaming query runs until it is stopped. Stopping it here also releases the checkpoint, so `spark_streaming_to_postgres.py` can be used afterwards.

In [23]:
query.stop()
print("Stream stopped. Still active:", query.isActive)

Stream stopped. Still active: False


In [24]:
spark.stop()